## Objective Function

Minimize Total Cost:
$$\text{Minimize } Z = B_{\text{total}} = \sum_{i \in I} c_i x_i$$

## Constraints

$$\sum_{i \in I} x_i \geq 5,650,000 \quad \text{(New Target Area)}$$

$$\sum_{i \in I} y_i \geq 25 \quad \text{(Geographic Equity)}$$

$$\sum_{i \in I_{arc}} c_i \cdot x_i \geq 61,750,000 \quad \text{(5-Year Arc of Deforestation Minimum)}$$

$$c_i \cdot x_i \leq \theta \cdot B_{\text{total}} \quad \forall i \in I \quad \text{(Dynamic Budget Share)}$$

$$x_i \leq A_i \cdot y_i \quad \forall i \in I \quad \text{(Eligible Land Limit)}$$

$$x_i \geq m \cdot y_i \quad \forall i \in I \quad \text{(Minimum Viable Scale)}$$

$$x_i \geq 0, \quad y_i \in \{0,1\} \quad \forall i \in I \quad \text{(Domain Bounds)}$$

In [11]:
from math import floor

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pulp

# load the data
df = pd.read_csv("Data\\MPAA_model_data.csv")

df['carbon index norm'] = df['carbon index norm'].fillna(df['carbon index norm'].mean())

# List of municipalities IDs
municipalities = df['NM_MUN'].tolist()

# Biodiversity score for each municipality Bio_i
Bio = df.set_index('NM_MUN')['biodiversity_priority_index'].to_dict()

# Extinction risk normalised E_hat_i
E_hat = df.set_index('NM_MUN')['Extinction risk score norm'].to_dict()

# Carbon sequestration potential for each municipality Car_hat_i
Car_hat = df.set_index('NM_MUN')['carbon index norm'].to_dict()

# Land available for planting in each municipality A_i
A = df.set_index('NM_MUN')['eligible_area_ha_mapbiomas'].to_dict()

# Urgency score U_hat_i
U_hat = df.set_index('NM_MUN')['urgency_5yr_index'].to_dict()

# Reversal risk Rev_hat_i
Rev_hat = df.set_index('NM_MUN')['reversal_risk'].to_dict()

# cost per hectare for each municipality c_i
c = df.set_index('NM_MUN')['cost_per_ha'].to_dict()

# max cost per hectare across all municipalities (for cost-effectiveness)
c_max = max(c.values())
c_max

# ========================= Fixed Additional Parameters ======================== 
#B = 489850000  # Total budget in dollars
m = 200    # Minimum viable project scale in hectares
K = 560000 # Updated based on new data and assumptions - can implement up to 560,000 hectares per year across all municipalities
N_min = 25       # Minimum number of municipalities to fund (Fairness constraint)
theta = 0.20  # No municipality can receive more than 20% of the total budget B

# weight constraints
alpha_B = 0.5      # Weight for Biodiversity
beta_B = 0.5       # Weight for Extinction Risk


 # ======================== New Parameters for Budget Min Problem ========================
TARGET_AREA = 5650000  # Target area to restore in hectares


In [13]:
# INTEGRAL
arc_municipalities = [
    "Abaetetuba",
    "Abel Figueiredo",
    "Acará",
    "Ananindeua",
    "Aurora do Pará",
    "Bagre",
    "Baião",
    "Barcarena",
    "Belém",
    "Benevides",
    "Bom Jesus do Tocantins",
    "Brejo Grande do Araguaia",
    "Breu Branco",
    "Bujaru",
    "Cametá",
    "Canaã dos Carajás",
    "Conceição do Araguaia",
    "Concórdia do Pará",
    "Curionópolis",
    "Eldorado do Carajás",
    "Floresta do Araguaia",
    "Goianésia do Pará",
    "Igarapé-Miri",
    "Inhangapi",
    "Irituia",
    "Itupiranga",
    "Jacundá",
    "Limoeiro do Ajuru",
    "Mãe do Rio",
    "Marituba",
    "Mocajuba",
    "Moju",
    "Nova Ipixuna",
    "Oeiras do Pará",
    "Palestina do Pará",
    "Piçarra",
    "Salvaterra",
    "Santa Bárbara do Pará",
    "Santa Izabel do Pará",
    "São Domingos do Araguaia",
    "São Domingos do Capim",
    "São Geraldo do Araguaia",
    "São João do Araguaia",
    "Sapucaia",
    "Tailândia",
    "Tomé-Açu",
    "Tucuruí",
    "Xinguara",
    # PARCIAL
    "Água Azul do Norte",
    "Anajás",
    "Anapu",
    "Bannach",
    "Bonito",
    "Breves",
    "Cachoeira do Arari",
    "Castanhal",
    "Chaves",
    "Colares",
    "Cumaru do Norte",
    "Curralinho",
    "Gurupá",
    "Marabá",
    "Melgaço",
    "Muaná",
    "Novo Repartimento",
    "Ourém",
    "Ourilândia do Norte",
    "Pacajá",
    "Parauapebas",
    "Pau D'Arco",
    "Ponta de Pedras",
    "Portel",
    "Porto de Moz",
    "Redenção",
    "Rio Maria",
    "Santa Cruz do Arari",
    "Santa Luzia do Pará",
    "Santa Maria das Barreiras",
    "Santana do Araguaia",
    "Santo Antônio do Tauá",
    "São Caetano de Odivelas",
    "São Félix do Xingu",
    "São Francisco do Pará",
    "São Miguel do Guamá",
    "São Sebastião da Boa Vista",
    "Senador José Porfírio",
    "Soure",
    "Vigia",
    # "Gurupi e Tocantins" — partially in your basin of interest
    "Capitão Poço",
    "Dom Eliseu",
    "Garrafão do Norte",
    "Ipixuna do Pará",
    "Nova Esperança do Piriá",
    "Paragominas",
    "Rondon do Pará",
    "Ulianópolis",
    "Viseu"]

In [12]:
import geopandas as gpd
municipalities_gdf = gpd.read_file("Data\\BR_Municipios_2024.shp")

# Filter Pará and project to metric CRS (meters
para_gdf = municipalities_gdf[municipalities_gdf["NM_UF"] == "Pará"].copy()
    
len(para_gdf)

144

## Budget Minimisation Problem

In [16]:
model = pulp.LpProblem("Minimum_Budget_5_65M_Goal", pulp.LpMinimize)

# --- 4. Decision Variables ---
x = pulp.LpVariable.dicts("x", municipalities, lowBound=0, cat=pulp.LpContinuous)
y = pulp.LpVariable.dicts("y", municipalities, cat=pulp.LpBinary)

# Auxiliary variable to track total budget so we can apply the theta constraint dynamically
Total_Budget = pulp.LpVariable("Total_Budget", lowBound=0, cat=pulp.LpContinuous)

# --- 5. Constraints ---
# Link the auxiliary variable to the actual costs
model += Total_Budget == pulp.lpSum(c[i] * x[i] for i in municipalities), "Define_Total_Budget"

# Target Area Constraint (The new driving force)
model += pulp.lpSum(x[i] for i in municipalities) >= TARGET_AREA, "Target_5_65M_Hectares"

# Geographic Equity
model += pulp.lpSum(y[i] for i in municipalities) >= N_min, "Min_Municipalities"

# Arc of Deforestation Minimum
model += pulp.lpSum(c[i] * x[i] for i in arc_municipalities) >= 74100000, "Arc_Minimum"

# Logical Bounds
for i in municipalities:
    model += x[i] <= A[i] * y[i], f"Upper_Bound_Area_{i}"
    model += x[i] >= m * y[i], f"Min_Viable_Scale_{i}"
    
    # Dynamic Maximum Budget Share (theta * B is now theta * Total_Budget)
    model += c[i] * x[i] <= theta * Total_Budget, f"Max_Budget_Share_{i}"

# --- 6. Objective ---
model += Total_Budget, "Minimize_Total_Cost"

# --- 7. Solve ---
print("Solving the Minimum Budget Model for 5.65M Hectares...")
model.solve(pulp.GUROBI(msg=True))

status = pulp.LpStatus[model.status]
print(f"\nModel Status: {status}")

if status == "Optimal":
    optimal_budget = pulp.value(Total_Budget)
    total_area_achieved = sum(x[i].varValue for i in municipalities)
    
    print("\n" + "="*50)
    print("CONCLUSION SCENARIO: MINIMUM BUDGET FOR 5.65M HA")
    print("="*50)
    print(f"Required Minimum Budget: US$ {optimal_budget:,.2f}")
    print(f"Total Area Restored:     {total_area_achieved:,.2f} ha")
    print(f"Average Cost per ha:     US$ {optimal_budget / total_area_achieved:,.2f}")
    print("="*50)
else:
    print("The model could not find an optimal solution. Ensure that the total eligible area sum(A_i) is actually >= 5,650,000.")

Solving the Minimum Budget Model for 5.65M Hectares...
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i5-1235U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 436 rows, 289 columns and 1394 nonzeros (Min)
Model fingerprint: 0xc4ce22bf
Model has 1 linear objective coefficients
Variable types: 145 continuous, 144 integer (0 binary)
Coefficient statistics:
  Matrix range     [2e-01, 2e+06]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+01, 7e+07]

Presolve removed 146 rows and 4 columns
Presolve time: 0.00s
Presolved: 290 rows, 285 columns, 1094 nonzeros
Variable types: 143 continuous, 142 integer (142 binary)

Root relaxation: objective 1.167197e+10, 126 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  